# Generate synthetic data from schema

Load a schema (from config or fixture), then generate a Spark DataFrame using **Databricks Labs Data Generator** (dbldatagen) via `schema_parser`.

- **Spark**: Use the injected `spark` session (Databricks Connect or local).
- **Schema**: Pipeline YAML, SDV, YData, or MySQL/PostgreSQL DDL (e.g. fixture from ronaldbradford/schema, postgresDBSamples, or neondatabase/postgres-sample-dbs).

In [ ]:
# Project root and connection check
import sys
from pathlib import Path

# Repo root: walk up until we find a directory containing both "src" and "tests"
cwd = Path.cwd()
root = cwd
while root != root.parent:
    if (root / "src").is_dir() and (root / "tests").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Project root:", root)

# Verify spark is available (injected by Databricks Connect or kernel)
try:
    print("Spark:", spark)
    spark.sql("SELECT 1 as one").show()
except NameError:
    print("No 'spark' in scope. Use a kernel with Databricks Connect or create a SparkSession.")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.schema_parser import load_schema, SchemaSource
from src.schema_parser.dbldatagen_builder import build_dataframe_from_canonical

# Schema path: use a fixture or config file
# Option A: MySQL Sakila excerpt (ronaldbradford/schema)
schema_path = root / "tests" / "fixtures" / "ronaldbradford_schema" / "sakila_excerpt.sql"
format_hint = SchemaSource.MYSQL

# Option B: Postgres Pagila excerpt (postgresDBSamples)
# schema_path = root / "tests" / "fixtures" / "postgresDBSamples" / "pagila_excerpt.sql"
# format_hint = SchemaSource.POSTGRES

# Option C: Pipeline or SDV/YData YAML from config
# schema_path = root / "config" / "pipeline_config.yaml"
# format_hint = None  # auto-detect

tables = load_schema(schema_path, format_hint=format_hint)
print(f"Loaded {len(tables)} table(s):", [t.name for t in tables])

In [ ]:
# Generate data for one or all tables
ROWS = 1000
SEED = 42
PARTITIONS = 8

dataframes = {}
for table in tables:
    df = build_dataframe_from_canonical(
        spark,
        table,
        rows=ROWS,
        partitions=PARTITIONS,
        seed=SEED,
    )
    dataframes[table.name] = df
    print(f"{table.name}: {df.count()} rows, {len(df.columns)} columns")

In [ ]:
# Show sample from first table
first_table = tables[0].name
df = dataframes[first_table]
df.printSchema()
print("Sample:")
df.show(5, truncate=20)